# RénoSim — Démo Phase 1 : moteur cœur

Décrire une maison → consommation, CO₂, étiquette DPE, dans les **deux modes**
(conventionnel et personnalisé). Le moteur `renosim` implémente une version simplifiée
de la méthode 3CL-DPE 2021 (écarts documentés dans `docs/deviations.md`).

**Cas d'étude** : maison des années 1960-70, 100 m², non isolée, chaudière fioul
d'origine, zone climatique H1a (nord-est), altitude < 400 m.

In [1]:
from renosim import (
    AltitudeClass,
    Building,
    ClimateZone,
    ConstructionPeriod,
    DHWGeneratorType,
    DHWSystem,
    EnergyCarrier,
    HeatingGeneratorType,
    HeatingSystem,
    OccupancyScenario,
    OpaqueEnvelopeElement,
    VentilationSystem,
    VentilationType,
    Window,
    simulate,
)
from renosim.envelope import default_u_values

# U par défaut 3CL pour une maison 1948-1974 non isolée
u = default_u_values(
    ConstructionPeriod.Y1948_1974,
    ClimateZone.H1A,
    insulated=False,
    electric_joule=False,
)
print(f"U par défaut : murs {u.wall}, toiture {u.roof}, plancher {u.floor} W/(m².K)")

U par défaut : murs 2.5, toiture 2.5, plancher 2.0 W/(m².K)


In [2]:
house = Building(
    living_area_m2=100.0,
    construction_period=ConstructionPeriod.Y1948_1974,
    climate_zone=ClimateZone.H1A,
    altitude_class=AltitudeClass.LOW,
    walls=(OpaqueEnvelopeElement(area_m2=120.0, u_value_w_per_m2k=u.wall),),
    roof=OpaqueEnvelopeElement(area_m2=100.0, u_value_w_per_m2k=u.roof),
    floor=OpaqueEnvelopeElement(area_m2=100.0, u_value_w_per_m2k=u.floor),
    windows=(Window(area_m2=15.0, u_value_w_per_m2k=4.8, solar_factor=0.56),),
    heating_system=HeatingSystem(
        generator_type=HeatingGeneratorType.STANDARD_BOILER,
        energy_carrier=EnergyCarrier.FUEL_OIL,
        generator_age_years=50,
    ),
    dhw_system=DHWSystem(
        generator_type=DHWGeneratorType.COUPLED_TO_HEATING_SYSTEM,
        energy_carrier=EnergyCarrier.FUEL_OIL,
    ),
    ventilation_system=VentilationSystem(ventilation_type=VentilationType.NATURAL),
)

## Mode conventionnel (celui de l'étiquette DPE)

Scénario normalisé : consigne 19 °C, occupation dérivée de la surface, 56 l/j/adulte d'ECS.

In [3]:
result = simulate(house)  # scénario conventionnel par défaut

print("Déperditions GV :", round(result.envelope.gv_w_per_k), "W/K")
print(
    "  dont murs",
    round(result.envelope.walls_w_per_k),
    "| toiture",
    round(result.envelope.roof_w_per_k),
    "| plancher",
    round(result.envelope.floor_w_per_k),
    "| fenêtres",
    round(result.envelope.windows_w_per_k),
    "| ponts th.",
    round(result.envelope.thermal_bridges_w_per_k),
    "| air",
    round(result.envelope.ventilation_w_per_k + result.envelope.infiltration_w_per_k),
    "W/K",
)
print()
print(
    "Besoins chauffage :",
    round(result.needs.heating_kwh),
    "kWh/an",
    "| ECS :",
    round(result.needs.dhw_kwh),
    "kWh/an",
)
print("Énergie finale   :", round(result.final_energy_kwh_m2), "kWh/m²/an")
print("Énergie primaire :", round(result.primary_energy_kwh_m2), "kWhep/m²/an")
print("Émissions        :", round(result.co2_kg_m2, 1), "kgCO₂/m²/an")
print("Coût annuel      :", round(result.annual_cost_eur), "€/an")
print()
print(
    ">>> ÉTIQUETTE DPE :",
    result.label,
    f"(énergie {result.energy_class} / climat {result.climate_class})",
)

Déperditions GV : 995 W/K
  dont murs 300 | toiture 250 | plancher 200 | fenêtres 72 | ponts th. 66 | air 107 W/K

Besoins chauffage : 55447 kWh/an | ECS : 1293 kWh/an
Énergie finale   : 636 kWh/m²/an
Énergie primaire : 639 kWhep/m²/an
Émissions        : 205.3 kgCO₂/m²/an
Coût annuel      : 5864 €/an

>>> ÉTIQUETTE DPE : G (énergie G / climat G)


## Mode personnalisé (« pour votre usage »)

Ménage économe : consigne 18 °C, 2 occupants. **L'étiquette DPE ne change pas** —
elle est toujours calculée avec le scénario conventionnel ; seuls les kWh, € et CO₂
« pour votre usage » bougent.

In [4]:
custom = simulate(house, OccupancyScenario(heating_setpoint_c=18.0, occupants_override=2.0))

print(
    "Énergie finale   :",
    round(custom.final_energy_kwh_m2),
    "kWh/m²/an",
    f"(conventionnel : {round(result.final_energy_kwh_m2)})",
)
print(
    "Coût annuel      :",
    round(custom.annual_cost_eur),
    "€/an",
    f"(conventionnel : {round(result.annual_cost_eur)})",
)
print(
    "CO₂              :",
    round(custom.co2_kg_m2, 1),
    "kgCO₂/m²/an",
    f"(conventionnel : {round(result.co2_kg_m2, 1)})",
)
print()
print(
    "Étiquette affichée = celle du mode conventionnel :",
    result.label,
    "(le résultat personnalisé n'est pas une étiquette : is_conventional =",
    str(custom.is_conventional) + ")",
)

Énergie finale   : 573 kWh/m²/an (conventionnel : 636)
Coût annuel      : 5288 €/an (conventionnel : 5864)
CO₂              : 184.9 kgCO₂/m²/an (conventionnel : 205.3)

Étiquette affichée = celle du mode conventionnel : G (le résultat personnalisé n'est pas une étiquette : is_conventional = False)


## Aperçu Phase 2

Les gestes de rénovation (`renovation.py`) transformeront ce `Building` immuable en
variantes rénovées (isolation, PAC, fenêtres, VMC…) avec coûts et temps de retour.